# SEG-FL 运行 notebook

每一格都可以单独运行（各格开头都会先进入项目目录），不依赖前面的格子，重启内核后也不用从头跑。

- 数据准备和实验都在后台运行，关掉网页或重启内核都不会中断
- 停止后重新运行对应的“启动”格即可续跑
- 服务器日志里的时间是北京时间

## 0. 首次使用：解压（已经解压过就跳过）

In [ ]:
%cd /root/autodl-tmp
!python3 -m zipfile -e SEG-FL.zip .

## 1. 状态与环境检查

In [ ]:
%cd /root/autodl-tmp/SEG-FL
!bash start.sh status
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv
!df -h /root/autodl-tmp | tail -1
!ls -lh pretrained dataset/voc_seg 2>/dev/null

## 2. 准备数据（只需一次，后台运行）

下载 VOC 2012（约 2 GB）和增强标注 SegmentationClassAug，生成 `dataset/voc_seg/*.npy`。
如果下载失败，把 `VOCtrainval_11-May-2012.tar` 和 `SegmentationClassAug.zip` 上传到 `dataset/raw/` 再运行这一格。
用第 4 格看进度，最后一行出现 `data ready` 就好了。

In [ ]:
%cd /root/autodl-tmp/SEG-FL
!PIP_INDEX_URL=https://pypi.tuna.tsinghua.edu.cn/simple bash start.sh prepare

## 2b. 学习率探测（试点前跑一次，后台运行）

无攻击、不算 Shapley、IID、seed 42，每个学习率跑一遍 25 轮（默认 0.01 / 0.005 / 0.0025 / 0.001）。
跑完后运行下一格：表里 `round0_share_of_loss_drop` 是第 0 轮占 25 轮总损失下降的比例，越小说明收敛越不集中在第 0 轮。
最后一行会给出建议的 `LOCAL_LR`，填进第 3 格和第 6 格。

In [ ]:
%cd /root/autodl-tmp/SEG-FL
!bash start.sh lr-probe

In [ ]:
%cd /root/autodl-tmp/SEG-FL
!PYTHONPATH=/root/autodl-tmp/SEG-FL/pip_pkgs python3 lr_probe_report.py results/lr_probe
from IPython.display import Image, display
import os
if os.path.exists("results/lr_probe/lr_probe.png"):
    display(Image("results/lr_probe/lr_probe.png"))

## 3. 启动试点（8 个小实验，后台运行）

seed 42；IID 和 Dirichlet α=0.5；无攻击 + DFR/SDFR/AFR 各 30%。看到 `started pilot (pid …)` 就是启动成功。
`LOCAL_LR` 改成第 2b 格建议的值。如果试点仍显示第 0 轮收敛太快，再把 `LOCAL_EPOCHS` 改成 1。

In [ ]:
%cd /root/autodl-tmp/SEG-FL
!PIP_INDEX_URL=https://pypi.tuna.tsinghua.edu.cn/simple LOCAL_EPOCHS=2 LOCAL_LR=0.005 bash start.sh pilot

## 4. 查看进度（随时运行）

In [ ]:
%cd /root/autodl-tmp/SEG-FL
!bash start.sh status

## 5. 查看试点结果（运行中也可以看已完成部分）

AUC 表、5 种分组对照、测试 mIoU、每轮耗时，以及“无攻击时 mIoU 随轮次”（判断是否收敛太快）。

In [ ]:
%cd /root/autodl-tmp/SEG-FL
!PYTHONPATH=/root/autodl-tmp/SEG-FL/pip_pkgs python3 summarize.py --results-root results/pilot > /dev/null
!PYTHONPATH=/root/autodl-tmp/SEG-FL/pip_pkgs python3 show_summary.py results/pilot
import glob
from IPython.display import Image, display
for p in sorted(glob.glob("results/pilot/plots/*.png")):
    print(p); display(Image(p))

## 6. 启动正式实验（200 个小实验，4 个划分并行，后台运行）

看到 `started full-parallel (pid …)` 就是启动成功。停止后再运行这一格会从断点续跑。

In [ ]:
%cd /root/autodl-tmp/SEG-FL
!PIP_INDEX_URL=https://pypi.tuna.tsinghua.edu.cn/simple LOCAL_EPOCHS=2 LOCAL_LR=0.005 bash start.sh full-parallel

## 7. 查看正式实验结果（运行中也可以看已完成部分）

In [ ]:
%cd /root/autodl-tmp/SEG-FL
!PYTHONPATH=/root/autodl-tmp/SEG-FL/pip_pkgs python3 summarize.py --results-root results/full_experiment > /dev/null
!PYTHONPATH=/root/autodl-tmp/SEG-FL/pip_pkgs python3 show_summary.py results/full_experiment
import glob
from IPython.display import Image, display
for p in sorted(glob.glob("results/full_experiment/plots/*.png")):
    print(p); display(Image(p))

## 8. 查看某一个小实验的 8 张图

先运行一次列出所有已完成的小实验，再把 `PICK` 改成想看的编号。

In [ ]:
%cd /root/autodl-tmp/SEG-FL
import glob, os
from IPython.display import Image, display
runs = sorted(os.path.dirname(p) for p in glob.glob("results/**/runs/*/summary.json", recursive=True))
for i, r in enumerate(runs):
    print(i, r)
PICK = -1   # 改成编号；-1 = 最后一个
if runs:
    for p in sorted(glob.glob(os.path.join(runs[PICK], "plots", "*.png"))):
        print(p); display(Image(p))

## 9. 打包结果下载

打包后在左侧文件栏进入 `autodl-tmp/SEG-FL`，右键 zip 选 Download。

In [ ]:
%cd /root/autodl-tmp/SEG-FL
import time
name = f"seg_results_{time.strftime('%Y%m%d_%H%M')}.zip"
!python3 -m zipfile -c {name} results logs
!ls -lh {name}

## 10. 停止

停止后重新运行第 3 格（试点）或第 6 格（正式）即可续跑。

In [ ]:
%cd /root/autodl-tmp/SEG-FL
!bash start.sh stop